# Point Predictors
### Notebook initialisation

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)


import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper")

### Dataset loading:

In [ ]:
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

from shared.gathered_robot_data import GatheredRobotData
from pose_estimation import HeadsetData, create_robot_bound_headset_data
from pose_estimation import RobotEnvironment, visualize_robot_camera_environment_combo, XYZImageGenerationConfig, ICPAlignmentConfig

robot_data = GatheredRobotData.from_folder(robot_data_folder_location)
robot_env = RobotEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data = create_robot_bound_headset_data(
        headset_data = HeadsetData.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)

## Hyperparameters

In [ ]:
from pose_estimation import OnlyPointsPredictor, ExtractAndMatchWrapperConfig, PredictionOnDataset, FastGrippingError

default_point_predictor = OnlyPointsPredictor(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig()
)

PredictionOnDataset(
    predictor = default_point_predictor,
    headset_data = labeled_headset_data, 
    gripping_error=FastGrippingError(points=robot_env.robot_xyz_images, intrinsics=labeled_headset_data.intrinsic_cam_mtx, visualize=False)
).print_summary()

## Extract and Match Options

The following `ExtractAndMatch` options are available: 

In [ ]:
from pose_estimation import GradablePosePredictor, NPredictors1DatasetGrader
from pose_estimation import ExtractAndLightGlue, ExtractAndMatchLoMa, ExtractAndMatchEffLoFTR


lightglue_variants = [
        GradablePosePredictor(
            creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndLightGlue(extractor=extractor)
                ),
            ),
            category = "LightGlue",
            name=f"{extractor} + LightGlue"
        )
        for extractor in ['SuperPoint', 'DISK', 'SIFT', 'ALIKED', 'DogHardNet']
]

loma_variants = [
    GradablePosePredictor(
        creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndMatchLoMa(loma_variant=variant)
                ),
            ),
        category = "LoMa",
        name=variant
    )
    for variant in ['LoMaB', 'LoMaB128', 'LoMaL', 'LoMaG', 'LoMaR']
]


# So bad it messes up the plot scaling
#loftr = GradablePosePredictor(
#    creator=OnlyPointsPredictor.get_creation_function(
#        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
#        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
#            extract_and_match=ExtractAndMatchEffLoFTR(matching_threshhold=0.7)
#        ),
#    ),
#    name="LoFTR"
#)


to_grade_predictors = lightglue_variants+loma_variants
different_matcher_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=to_grade_predictors,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
import matplotlib.pyplot as plt
from pose_estimation import SingleValueErrorType

different_matcher_grades.print_summary()

fig, axes = plt.subplots(4,1, figsize = (20, 18))
different_matcher_grades.plot_hz_vs_error(ax=axes[0], error_type=SingleValueErrorType.MED_TRANSLATIONAL, plot_frontier=False)
different_matcher_grades.plot_hz_vs_error(ax=axes[1], error_type=SingleValueErrorType.MED_ROTATIONAL, plot_frontier=False)
different_matcher_grades.plot_hz_vs_error(ax=axes[2], error_type=SingleValueErrorType.AVG_GRIPPING_ERROR, plot_frontier=False)
different_matcher_grades.plot_hz_vs_error(ax=axes[3], error_type=SingleValueErrorType.AVG_NUMBER_POINTS_INLIERS, plot_frontier=False, invert_y=False)


## Augmentations
### Rotation Augmentations

In [ ]:
from pose_estimation import Augmentation, Rotate180Deg

rot_augmentation_options = [[Augmentation], [Rotate180Deg], [Augmentation, Rotate180Deg]]


super_points = [
    GradablePosePredictor(
            creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SuperPoint"),
                rotation_augmentations=augs
            ),
        ),
        name=f"SuperPoint + LightGlue, {[str(aug()) for aug in augs]}"
    )
    for augs in rot_augmentation_options
]

sifts = [
    GradablePosePredictor(
            creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SIFT"),
                rotation_augmentations=augs
            ),
        ),
        name=f"SIFT + LightGlue, {[str(aug()) for aug in augs]}"
    )
    for augs in rot_augmentation_options
]

lomas = [
    GradablePosePredictor(
        creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndMatchLoMa(loma_variant="LoMaB128"),
                    rotation_augmentations=augs
                ),
            ),
        name=f"LoMaB128, {[str(aug()) for aug in augs]}"
    )
    for augs in rot_augmentation_options
]


different_rot_augmentations_predictors = super_points + sifts + lomas
different_rot_aug_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_rot_augmentations_predictors,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
import matplotlib.pyplot as plt

different_rot_aug_grades.print_summary()

fig, axes = plt.subplots(2,1, figsize = (16, 12))
different_rot_aug_grades.plot_hz_vs_error(ax=axes[0], error_type=SingleValueErrorType.MED_TRANSLATIONAL, plot_frontier=False)
different_rot_aug_grades.plot_hz_vs_error(ax=axes[1], error_type=SingleValueErrorType.MED_ROTATIONAL, plot_frontier=False)


### Crop Augmentations

In [ ]:
lomas_crop = [
    GradablePosePredictor(
        creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndMatchLoMa(loma_variant="LoMaB128"),
                    rotation_augmentations=[Rotate180Deg],
                    crop_augmentations=[amount] if amount is not None else None
                ),
            ),
        category = "LoMaB128",
        name=f"Crop: {amount}"
    )
    for amount in [None, 0.3, 0.4, 0.5, 0.6]
]

super_points_crop = [
    GradablePosePredictor(
            creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SuperPoint"),
                rotation_augmentations=[Rotate180Deg],
                crop_augmentations=[amount] if amount is not None else None
            ),
        ),
        category = "LightGlue",
        name=f"Crop: {amount}"
    )
    for amount in [None, 0.3, 0.4, 0.5, 0.6]
]

different_crop_augmentations_predictors = lomas_crop + super_points_crop
different_crop_aug_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_crop_augmentations_predictors,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
import matplotlib.pyplot as plt

different_crop_aug_grades.print_summary()

fig, axes = plt.subplots(2,1, figsize = (16, 12))
different_crop_aug_grades.plot_hz_vs_error(ax=axes[0], error_type=SingleValueErrorType.MED_TRANSLATIONAL, plot_frontier=False, use_category=True)
different_crop_aug_grades.plot_hz_vs_error(ax=axes[1], error_type=SingleValueErrorType.MED_ROTATIONAL, plot_frontier=False, use_category=True)

fig, axes1 = plt.subplots(2,1, figsize = (16, 12))
different_crop_aug_grades.plot_hz_vs_error(ax=axes1[0], error_type=SingleValueErrorType.AVG_TRANSLATIONAL, plot_frontier=False, use_category=True)
different_crop_aug_grades.plot_hz_vs_error(ax=axes1[1], error_type=SingleValueErrorType.AVG_ROTATIONAL, plot_frontier=False, use_category=True)

## Schedulers
Schedulers will switch through the robot images if a pose cant be predicted

In [ ]:
from pose_estimation import Scheduler, EMAScheduler, BlockingEMAScheduler
from pose_estimation import RansacPoseEstimationConfig

super_points_shedulers = [
    GradablePosePredictor(
            creator=OnlyPointsPredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(extractor="SuperPoint"),
                rotation_augmentations=[Rotate180Deg],
                ransac_config = RansacPoseEstimationConfig(min_number_inlier_afterwards = 35),
                scheduler=scheduler
            ),
        ),
        name=f"{scheduler(1)}"
    )
    for scheduler in [Scheduler, EMAScheduler, BlockingEMAScheduler]
]


different_scheduler_aug_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=super_points_shedulers,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
from pose_estimation import TimeSeriesErrorType

import matplotlib.pyplot as plt

different_scheduler_aug_grades.print_summary()

fig, axes = plt.subplots(2,1, figsize = (16, 12))
different_scheduler_aug_grades.plot_time_series_error(ax=axes[0], error_type=TimeSeriesErrorType.ABS_ROTATIONAL)
different_scheduler_aug_grades.plot_time_series_error(ax=axes[1], error_type=TimeSeriesErrorType.ABS_TRANSLATIONAL)
